In [1]:
import pandas as pd
from pathlib import Path

In [2]:
pd.set_option("display.max_columns", None)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

# 02 - Data Cleaning
Clean the merged table:

- Keep delivered orders

- Handle missing values and duplicates

- Fix data types, and check categorical and numerical values.

In [3]:
df = pd.read_csv(PROCESSED / "orders_merged.csv")
steps = {"merged": len(df)}
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")

112,647 rows x 30 columns


# 1. Filter Delivered Orders
Only **delivered orders** have complete delivery dates, so the other statuses are removed.

In [4]:
df["order_status"].value_counts()

order_status
delivered      110194
shipped          1185
canceled          542
invoiced          359
processing        357
unavailable         7
approved            3
Name: count, dtype: int64

In [5]:
df = df[df["order_status"] == "delivered"].copy()
steps["delivered"] = len(df)
print(f"{df.shape[0]:,} rows")

110,194 rows


# 2. Missing Values
Percentage of missing values per column. 

**Columns under 5% are deleted**

In [6]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
})
missing[missing["missing"] > 0]

,missing,missing_%
order_approved_at,15,0.01
order_delivered_carrier_date,2,0.00
order_delivered_customer_date,8,0.01
product_category_name,1537,1.39
product_name_lenght,1537,1.39
product_description_lenght,1537,1.39
product_photos_qty,1537,1.39
product_weight_g,18,0.02
product_length_cm,18,0.02
product_height_cm,18,0.02


In [7]:
df = df.dropna()
steps["no missing"] = len(df)
print(f"{df.shape[0]:,} rows")

108,633 rows


*Note: Most missing rows are products without category, which also have no name, description or photo data.*

# 3. Duplicates
Check for fully duplicated rows and for repeated `order_id` + `order_item_id` (the key of the table).

In [8]:
print("Duplicated rows:", df.duplicated().sum())
print("Duplicated keys:", df.duplicated(subset=["order_id", "order_item_id"]).sum())

Duplicated rows: 0
Duplicated keys: 0


# 4. Data Types
Convert the date columns to datetime and check that the dates follow a logical order.

## 4.1 Convert Dates

In [9]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

df[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
shipping_limit_date              datetime64[us]
dtype: object

## 4.2 Date Order Checks

In [10]:
checks = {
    "approved before purchase": df["order_approved_at"] < df["order_purchase_timestamp"],
    "carrier before purchase": df["order_delivered_carrier_date"] < df["order_purchase_timestamp"],
    "customer before carrier": df["order_delivered_customer_date"] < df["order_delivered_carrier_date"],
    "customer before purchase": df["order_delivered_customer_date"] < df["order_purchase_timestamp"],
}
pd.Series({name: mask.sum() for name, mask in checks.items()}, name="rows")

approved before purchase      0
carrier before purchase     185
customer before carrier      49
customer before purchase      0
Name: rows, dtype: int64

In [11]:
print("First purchase:", df["order_purchase_timestamp"].min())
print("Last purchase: ", df["order_purchase_timestamp"].max())

First purchase: 2016-10-03 09:44:50
Last purchase:  2018-08-29 15:00:37


*Note: Purchase and customer delivery dates are always consistent.** 

A few rows have a carrier date before the purchase or after the delivery, which looks like a recording error in that column.

# 5. Categorical Consistency
Check the values of the categorical columns and translate product categories to English.

## 5.1 Value Counts

In [12]:
for col in ["payment_type", "customer_state"]:
    display(df[col].value_counts())

print("Product categories:", df["product_category_name"].nunique())
print("Customer cities:   ", df["customer_city"].nunique())

payment_type
credit_card    83372
boleto         22047
debit_card      1624
voucher         1590
Name: count, dtype: int64

customer_state
SP    45809
RJ    13928
MG    12734
RS     6036
PR     5572
SC     4037
BA     3625
DF     2323
GO     2225
ES     2207
PE     1723
CE     1409
PA     1036
MT     1025
MS      802
MA      790
PB      577
PI      516
RN      515
AL      423
SE      373
TO      306
RO      267
AM      161
AC       89
AP       81
RR       44
Name: count, dtype: int64

Product categories: 73
Customer cities:    4073


## 5.2 Category Names in English
The translation table is missing two categories, which are added by hand.

In [13]:
translation = pd.read_csv(RAW / "product_category_name_translation.csv")
extra = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_and_food_preparers"],
})
translation = pd.concat([translation, extra], ignore_index=True)

df = df.merge(translation, on="product_category_name", how="left")
print("Untranslated rows:", df["product_category_name_english"].isna().sum())
df["product_category_name_english"].value_counts().head(10)

Untranslated rows: 0


product_category_name_english
bed_bath_table           10952
health_beauty             9462
sports_leisure            8428
furniture_decor           8156
computers_accessories     7643
housewares                6795
watches_gifts             5857
telephony                 4428
garden_tools              4267
auto                      4139
Name: count, dtype: int64

# 6. Numerical Ranges
Check that numerical values are in a logical range. 

## 6.1 Summary

In [14]:
num_cols = [
    "price", "freight_value", "payment_value", "payment_installments", "n_payments",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
    "product_photos_qty", "product_name_lenght", "product_description_lenght",
]
df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
price,108633.0,120.089235,182.046331,0.85,39.90,74.90,134.90,6735.00
freight_value,108633.0,19.983410,15.733777,0.00,13.08,16.29,21.16,409.68
payment_value,108633.0,179.776077,271.821441,9.59,65.71,114.27,195.44,13664.08
payment_installments,108633.0,3.020859,2.806026,0.00,1.00,2.00,4.00,24.00
n_payments,108633.0,1.043725,0.374852,1.00,1.00,1.00,1.00,26.00
product_weight_g,108633.0,2095.664982,3744.171572,0.00,300.00,700.00,1800.00,40425.00
product_length_cm,108633.0,30.198651,16.157603,7.00,18.00,25.00,38.00,105.00
product_height_cm,108633.0,16.588118,13.433232,2.00,8.00,13.00,20.00,105.00
product_width_cm,108633.0,23.030893,11.697321,6.00,15.00,20.00,30.00,118.00
product_photos_qty,108633.0,2.210866,1.721845,1.00,1.00,1.00,3.00,20.00


## 6.2 Impossible Values

In [15]:
impossible = (
    (df["price"] <= 0)
    | (df["product_weight_g"] <= 0)
    | (df["payment_installments"] < 1)
)
print("Weight <= 0:     ", (df["product_weight_g"] <= 0).sum())
print("Installments < 1:", (df["payment_installments"] < 1).sum())
print("Freight = 0:     ", (df["freight_value"] == 0).sum())

df = df[~impossible].copy()
steps["valid ranges"] = len(df)
print(f"{df.shape[0]:,} rows")

Weight <= 0:      8
Installments < 1: 3
Freight = 0:      379
108,622 rows


*Note*:
- Products with weight 0 and payments with 0 installments are data errors so they are removed. 

- Freight 0 is kept because it means free shipping.

# 7. Save
Save the clean table for the outliers notebook.

In [16]:
df.to_csv(PROCESSED / "orders_clean.csv", index=False)

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Orders:    {df['order_id'].nunique():,}")
print(f"Customers: {df['customer_unique_id'].nunique():,}")
pd.Series(steps, name="rows")

108,622 rows x 31 columns
Orders:    95,114
Customers: 92,067


merged          112647
delivered       110194
no missing      108633
valid ranges    108622
Name: rows, dtype: int64